In [400]:
import numpy as np
import torch
import torch.nn as nn
import copy
from scipy.sparse import coo_matrix

data = np.load('ddec_training_data.npz')
ddec_pts  = data['ddec_pts']
ddec_tris = data['ddec_tris']
ddec_mu   = float(data['ddec_mu'])
ddec_lam  = float(data['ddec_lam'])
loads_arr = data['loads']
U_all     = data['U_all']
P_all     = data['P_all']

ddec_solves = [{'load': tuple(loads_arr[i]), 'U': U_all[i], 'P': P_all[i]}
               for i in range(len(loads_arr))]

print(f"loaded {len(ddec_solves)} load cases, mu={ddec_mu}, lam={ddec_lam}")
print(f"ddec_pts shape: {ddec_pts.shape}, ddec_tris shape: {ddec_tris.shape}")

loaded 4 load cases, mu=1.0, lam=1000.0
ddec_pts shape: (16384, 2), ddec_tris shape: (32258, 3)


In [401]:
def get_batched_grads(pts, tris3):
    coords = pts[tris3]
    x1,y1 = coords[:,0,0], coords[:,0,1]
    x2,y2 = coords[:,1,0], coords[:,1,1]
    x3,y3 = coords[:,2,0], coords[:,2,1]
    areas = 0.5*np.abs((x2-x1)*(y3-y1) - (x3-x1)*(y2-y1))
    grads = np.zeros((len(tris3), 3, 2))
    grads[:,0,0]=(y2-y3)/(2*areas); grads[:,0,1]=(x3-x2)/(2*areas)
    grads[:,1,0]=(y3-y1)/(2*areas); grads[:,1,1]=(x1-x3)/(2*areas)
    grads[:,2,0]=(y1-y2)/(2*areas); grads[:,2,1]=(x2-x1)/(2*areas)
    return grads, areas

In [402]:
ddec_grads, ddec_areas = get_batched_grads(ddec_pts, ddec_tris)   # (n_tris,3,2), (n_tris,)


In [403]:
def scipy_to_torch_sparse(A):
    A = A.tocoo()
    idx = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)
    val = torch.tensor(A.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(idx, val, A.shape).coalesce()

In [404]:
def assemble_M_P1(areas, tris):
    """Global fine-scale P1 mass matrix: (M_P1)_ab = ∫ ω_a ω_b dx."""
    n_tris = len(tris)
    N = tris.max() + 1

    M_base = np.array([[2,1,1],[1,2,1],[1,1,2]]) / 12.0
    M_loc = areas[:,None,None] * M_base[None,:,:]          # (n_tris,3,3)

    rows = np.broadcast_to(tris[:,:,None], (n_tris,3,3)).reshape(-1)
    cols = np.broadcast_to(tris[:,None,:], (n_tris,3,3)).reshape(-1)

    return coo_matrix((M_loc.reshape(-1), (rows, cols)), shape=(N,N)).tocsr()

ddec_M_P1 = assemble_M_P1(ddec_areas, ddec_tris)
print(f"M_P1: shape={ddec_M_P1.shape}, nnz={ddec_M_P1.nnz}, symmetric={np.allclose((ddec_M_P1-ddec_M_P1.T).data, 0) if ddec_M_P1.nnz else True}")

M_P1: shape=(16384, 16384), nnz=113666, symmetric=True


In [405]:
N = len(ddec_pts)
local_edge_verts = [(1,2), (2,0), (0,1)]   # local edge k opposite local vertex k — same convention as build_p2_mesh

p = np.stack([ddec_tris[:,lp] for lp,lq in local_edge_verts], axis=1)   # (n_tris,3)
q = np.stack([ddec_tris[:,lq] for lp,lq in local_edge_verts], axis=1)   # (n_tris,3)

lo, hi = np.minimum(p,q), np.maximum(p,q)
edge_sign = np.where(p < q, 1.0, -1.0)                  # (n_tris,3) — orientation vs. canonical (lo,hi)
edge_key  = lo.astype(np.int64)*N + hi.astype(np.int64)  # (n_tris,3) — unique integer per edge

unique_keys, edge_gidx = np.unique(edge_key, return_inverse=True)   # keep unique_keys this time
edge_gidx = edge_gidx.reshape(edge_key.shape)            # (n_tris,3) — global edge index per local edge
n_edges = edge_gidx.max() + 1

lo_unique = (unique_keys // N).astype(np.int64)
hi_unique = (unique_keys %  N).astype(np.int64)


print(f"n_edges = {n_edges}")


n_edges = 48641


In [406]:
G    = np.einsum('tid,tjd->tij', ddec_grads, ddec_grads)         # (n_tris,3,3), G[t,i,j] = ∇λ_i·∇λ_j
Mloc = ddec_areas[:,None,None] * (np.array([[2,1,1],[1,2,1],[1,1,2]])/12.0)[None,:,:]   # (n_tris,3,3)

In [407]:
edge_p = np.array([1,2,0])   # matches local_edge_verts above
edge_q = np.array([2,0,1])

pp, qq = edge_p[:,None], edge_q[:,None]    # (3,1) — broadcasts over row k
rr, ss = edge_p[None,:], edge_q[None,:]    # (1,3) — broadcasts over col l
pp, qq, rr, ss = np.broadcast_arrays(pp, qq, rr, ss)   # all (3,3)

L = (G[:, qq, ss]*Mloc[:, pp, rr] - G[:, qq, rr]*Mloc[:, pp, ss]
    - G[:, pp, ss]*Mloc[:, qq, rr] + G[:, pp, rr]*Mloc[:, qq, ss])    # (n_tris,3,3)

In [ ]:
# dropout regularization
# SHAMPOO / SOAP

In [408]:
contrib = edge_sign[:,:,None] * edge_sign[:,None,:] * L     # (n_tris,3,3)

r_idx = np.broadcast_to(edge_gidx[:,:,None], (len(ddec_tris),3,3)).reshape(-1)
c_idx = np.broadcast_to(edge_gidx[:,None,:], (len(ddec_tris),3,3)).reshape(-1)

ddec_M_Ned = coo_matrix((contrib.reshape(-1), (r_idx, c_idx)), shape=(n_edges,n_edges)).tocsr()
print(f"M_Ned: shape={ddec_M_Ned.shape}, nnz={ddec_M_Ned.nnz}, symmetric={np.allclose((ddec_M_Ned-ddec_M_Ned.T).data, 0)}")

M_Ned: shape=(48641, 48641), nnz=242189, symmetric=True


In [409]:
def assemble_G(areas, grads, tris, d):
    """Fine-scale mixed mass-gradient matrix: G_d[a,b] = ∫ ω_a ∂ω_b/∂x_d dx.
    Per triangle, ∂λ_b/∂x_d is constant and ∫λ_a dx = area/3 for every vertex a,
    so the local (3,3) block is identical across rows a, varying only by column b.
    Deliberately asymmetric -- that's what makes the antisymmetric "oriented
    area" extraction below nonzero.
    """
    n_tris = len(tris)
    N = tris.max() + 1
    
    G_loc_col = (areas[:, None] / 3.0) * grads[:, :, d]             # (n_tris, 3) -- one value per column b
    G_loc = np.broadcast_to(G_loc_col[:, None, :], (n_tris, 3, 3))   # (n_tris,3,3), same across rows a
    
    rows = np.broadcast_to(tris[:, :, None], (n_tris, 3, 3)).reshape(-1)
    cols = np.broadcast_to(tris[:, None, :], (n_tris, 3, 3)).reshape(-1)
    
    return coo_matrix((G_loc.reshape(-1), (rows, cols)), shape=(N, N)).tocsr()
    
ddec_G_x = assemble_G(ddec_areas, ddec_grads, ddec_tris, d=0)
ddec_G_y = assemble_G(ddec_areas, ddec_grads, ddec_tris, d=1)

print(f"G_x: shape={ddec_G_x.shape}, nnz={ddec_G_x.nnz}, symmetric={np.allclose((ddec_G_x-ddec_G_x.T).data, 0)}")
print(f"G_y: shape={ddec_G_y.shape}, nnz={ddec_G_y.nnz}, symmetric={np.allclose((ddec_G_y-ddec_G_y.T).data, 0)}")

G_x: shape=(16384, 16384), nnz=113666, symmetric=False
G_y: shape=(16384, 16384), nnz=113666, symmetric=False


In [410]:
G_x_t = scipy_to_torch_sparse(ddec_G_x)
G_y_t = scipy_to_torch_sparse(ddec_G_y)

In [411]:
import torch
import torch.nn as nn

class POUGenerator(nn.Module):
    """Learns the interior partition-of-unity split via a softmax MLP.
    Boundary-node membership is a *fixed* geometric assignment (by y-band),
    not learned -- those partitions get clamped to a prescribed Dirichlet
    value downstream regardless of how nodes are split among them, so
    giving the optimizer freedom there only adds an unconstrained DOF that
    can warp M0/M1 conditioning without ever changing a prediction.
    """
    def __init__(self, n_interior=8, n_boundary=2, hidden=32, prior_scale=5.0):
        super().__init__()
        self.n_interior = n_interior
        self.n_boundary = n_boundary
        self.prior_scale = prior_scale
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, n_interior),
        )
        nn.init.zeros_(self.net[-1].weight)   # trainable part starts near-silent;
        nn.init.zeros_(self.net[-1].bias)     # the fixed prior below dominates at init

    def spatial_prior(self, xy, is_boundary):
        prior = torch.zeros(xy.shape[0], self.n_interior)
        x = xy[:, 0]
        int_band = torch.clamp((x * self.n_interior).long(), 0, self.n_interior - 1)
        rows = torch.arange(xy.shape[0])
        prior[rows[~is_boundary], int_band[~is_boundary]] = self.prior_scale
        return prior

    def boundary_membership(self, xy, is_boundary):
        """Fixed (non-trainable) hard assignment of boundary nodes to boundary
        partitions, by y-band -- analogous to W_NN_transformer's bc_pous_add_on."""
        membership = torch.zeros(xy.shape[0], self.n_boundary)
        y = xy[:, 1]
        bdry_band = torch.clamp((y * self.n_boundary).long(), 0, self.n_boundary - 1)
        rows = torch.arange(xy.shape[0])
        membership[rows[is_boundary], bdry_band[is_boundary]] = 1.0
        return membership

    def forward(self, xy, log_lam, is_boundary):
        n_fine = xy.shape[0]
        lam_col = log_lam.expand(n_fine, 1)
        inp = torch.cat([xy, lam_col], dim=1)
        logits = self.net(inp) + self.spatial_prior(xy, is_boundary)

        interior_w = torch.softmax(logits, dim=1)
        interior_w = torch.where(is_boundary[:, None], torch.zeros_like(interior_w), interior_w)
        boundary_w = self.boundary_membership(xy, is_boundary)

        return torch.cat([interior_w, boundary_w], dim=1)


In [412]:
N0_int, N0_bdry = 8, 2
xy = torch.tensor(ddec_pts, dtype=torch.float32)
is_bdry = torch.tensor(ddec_pts[:,0] < 1e-12)

pou_net = POUGenerator(n_interior=N0_int, n_boundary=N0_bdry)
test_log_lam = torch.tensor(np.log10(1000.0), dtype=torch.float32)
W = pou_net(xy, test_log_lam, is_bdry)

print("W shape:", W.shape)
print("row sums (should all be 1):", W.sum(dim=1).min().item(), W.sum(dim=1).max().item())
print("interior rows touching boundary cols (should be 0):", W[~is_bdry, N0_int:].abs().max().item())
print("boundary rows touching interior cols (should be 0):", W[is_bdry, :N0_int].abs().max().item())

W shape: torch.Size([16384, 10])
row sums (should all be 1): 0.9999998211860657 1.0
interior rows touching boundary cols (should be 0): 0.0
boundary rows touching interior cols (should be 0): 0.0


In [413]:
N0 = N0_int + N0_bdry
pairs_i, pairs_j = torch.triu_indices(N0, N0, offset=1)

def compute_oriented_areas(W, pairs_i, pairs_j):
    """Per-pair oriented direction vector between coarse partitions i and j.
    Built exactly like coarse_M1 -- coarsen a fixed fine-scale matrix via W,
    then antisymmetrize -- but using G_x/G_y (mixed mass-gradient) instead of
    M_Ned (edge mass), so the result is a *direction* instead of a magnitude.
    Mirrors the reference's compute_oriented_areas/compute_psi_1_integrals.
    """
    components = []
    for G_t in (G_x_t, G_y_t):
        A = W.T @ torch.sparse.mm(G_t, W)                   # (N0, N0)
        components.append(A[pairs_i, pairs_j] - A[pairs_j, pairs_i])
    return torch.stack(components, dim=1).abs()              # (n_pairs, 2)
    
oriented_areas_test = compute_oriented_areas(W, pairs_i, pairs_j)
print("oriented_areas shape:", oriented_areas_test.shape)
print("sample (first 5 pairs):\n", oriented_areas_test[:5])


oriented_areas shape: torch.Size([45, 2])
sample (first 5 pairs):
 tensor([[9.0580e-01, 5.8919e-10],
        [6.1032e-03, 6.9740e-10],
        [6.1032e-03, 6.9740e-10],
        [6.1032e-03, 6.9740e-10],
        [6.1032e-03, 6.9740e-10]], grad_fn=<SliceBackward0>)


In [414]:
import torch.nn as nn
import torch

class LightweightTransformerLayer(nn.Module):
    """Conditioning block: every token (one per coarse interface pair) gets the
    same additive term derived from z, then a per-token feedforward block.

    With exactly one key/value token, softmax attention is mathematically
    forced to weight 1.0 on that token regardless of the query -- an exact
    identity (softmax of a single logit is always 1), not an approximation.
    So the query/key projections in nn.MultiheadAttention never affect the
    output and never receive gradient; the attention output collapses to
    out_proj(v_proj(z)), broadcast identically to every token. We compute
    that closed form directly, which also sidesteps nn.MultiheadAttention's
    fused SDPA kernel -- the one the CPU/vmap batching-rule warning flagged,
    and the likely source of the NaNs under jacrev.
    """
    def __init__(self, d_model, num_heads=None, dropout_rate=0.0, activation=nn.Tanh(), use_norms=True):
        super().__init__()
        # num_heads accepted for call-site compatibility; unused now -- see docstring
        self.use_norms = use_norms
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), activation, nn.Dropout(dropout_rate),
            nn.Linear(d_model * 2, d_model),
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, z_embedding):
        # x: [n_pairs, d_model], z_embedding: [1, d_model]
        residual = x
        x_norm = self.norm1(x) if self.use_norms else x
        attn_output = self.out_proj(self.v_proj(z_embedding))   # (1, d_model) -- exact closed form, same for every token
        x = residual + self.dropout(attn_output)                # broadcasts (1,d_model) against (n_pairs,d_model)

        residual = x
        x_norm = self.norm2(x) if self.use_norms else x
        x = residual + self.dropout(self.ffn(x_norm))

        return x


    
class FluxCorrectionTransformer(nn.Module):
    """N(u_i, u_j, oriented_area, lambda; theta_N), attention-based -- mirrors
    the reference's PairwiseFlux_transformer. Embeds (u_i, u_j, oriented_area)
    per interface pair into a latent token, conditions via cross-attention to a
    single log10(lambda) token, projects back to a flux correction vector.

    Explicitly subtracts the network's output at (u_i=u_j=0) so a uniform
    rigid-body translation gives exactly zero correction -- same guarantee the
    plain-MLP FluxCorrectionNetwork had.
    """
    def __init__(self, num_fields=2, problem_dim=2, num_input_params=1,
                latent_dim=16, num_heads=4, num_blocks=2,
                dropout_rate=0.0, activation=nn.Tanh(), use_norms=True):
        super().__init__()
        self.u_embedding = nn.Linear(num_fields, latent_dim // 4)
        self.oriented_areas_embedding = nn.Linear(problem_dim, latent_dim // 2)
        self.z_embedding = nn.Linear(num_input_params, latent_dim)
        self.final_projection = nn.Linear(latent_dim, num_fields)
        self.transformer_blocks = nn.ModuleList(
            LightweightTransformerLayer(latent_dim, num_heads, dropout_rate, activation, use_norms)
            for _ in range(num_blocks)
        )   
        
    def _f(self, u_i, u_j, oriented_areas, log_lam):
        u_i_emb  = self.u_embedding(u_i)                          # (n_pairs, latent_dim//4)
        u_j_emb  = self.u_embedding(u_j)                          # (n_pairs, latent_dim//4)
        area_emb = self.oriented_areas_embedding(oriented_areas)  # (n_pairs, latent_dim//2)
        tokens   = torch.cat([u_i_emb, u_j_emb, area_emb], dim=1) # (n_pairs, latent_dim)
        
        z = self.z_embedding(log_lam.reshape(1, 1))                 # (1, latent_dim)
        for block in self.transformer_blocks:
            tokens = block(tokens, z)
            
        return self.final_projection(tokens)                        # (n_pairs, num_fields)
        
    def forward(self, u_i, u_j, oriented_areas, log_lam, zero_ref=None):
        raw = self._f(u_i, u_j, oriented_areas, log_lam)
        if zero_ref is None:
            zero = self._f(
                torch.zeros_like(u_i),
                torch.zeros_like(u_j),
                oriented_areas,
                log_lam
            )
        else:
            zero = zero_ref

        return raw - zero

In [415]:
N0 = 10
pairs_i, pairs_j = torch.triu_indices(N0, N0, offset=1)   # same 45 pairs as before
n_edges = pairs_i.shape[0]

delta_0 = torch.zeros(n_edges, N0)
delta_0[torch.arange(n_edges), pairs_j] =  1.0
delta_0[torch.arange(n_edges), pairs_i] = -1.0

print("delta_0 shape:", delta_0.shape)        # (45, 10)
print("row sums (should be 0 -- +1 and -1 per row):", delta_0.sum(dim=1).abs().max().item())
print("delta_0 for edge 0 (pair", (pairs_i[0].item(), pairs_j[0].item()), "):", delta_0[0])

delta_0 shape: torch.Size([45, 10])
row sums (should be 0 -- +1 and -1 per row): 0.0
delta_0 for edge 0 (pair (0, 1) ): tensor([-1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])


In [416]:
flux_net = FluxCorrectionTransformer(num_fields=2, problem_dim=2, num_input_params=1,
                                    latent_dim=16, num_heads=4, num_blocks=2)
u_coarse = torch.randn(N0, 2)       
log_lam  = torch.tensor(np.log10(1000.0), dtype=torch.float32)
oriented_areas_smoke = compute_oriented_areas(W, pairs_i, pairs_j)

flux = flux_net(
    u_coarse[pairs_i],
    u_coarse[pairs_j],
    oriented_areas_smoke,
    log_lam
)
print("flux shape:", flux.shape)

u_translate = torch.ones(N0, 2) * 7.3
flux_translate = flux_net(u_translate[pairs_i], u_translate[pairs_j], oriented_areas_smoke, log_lam)
print("flux under uniform translation (should be exactly 0):", flux_translate.abs().max().item())

flux shape: torch.Size([45, 2])
flux under uniform translation (should be exactly 0): 0.09179344773292542


In [417]:
import torch
import numpy as np

N0 = 10
pairs_i, pairs_j = torch.triu_indices(N0, N0, offset=1)
n_pairs = pairs_i.shape[0]
print(f"n_pairs = {n_pairs}")  # 45

u_coarse = torch.randn(N0, 2)

log_lam = torch.tensor(np.log10(1000.0), dtype=torch.float32)

# correct initialization (DO NOT pass log_lam)
flux_net = FluxCorrectionTransformer(
    num_fields=2,
    problem_dim=2,
    num_input_params=1,
    latent_dim=16,
    num_heads=4,
    num_blocks=2
)

# dummy geometry
oriented_areas = torch.randn(n_pairs, 2)

# forward test
flux = flux_net(
    u_coarse[pairs_i],
    u_coarse[pairs_j],
    oriented_areas,
    log_lam
)

print("flux shape:", flux.shape)

# translation test
u_translate = torch.ones(N0, 2) * 7.3

flux_translate = flux_net(
    u_translate[pairs_i],
    u_translate[pairs_j],
    oriented_areas,
    log_lam
)

print("max |flux| under translation:", flux_translate.abs().max().item())

n_pairs = 45
flux shape: torch.Size([45, 2])
max |flux| under translation: 1.4687962532043457


In [418]:
M_P1_t  = scipy_to_torch_sparse(ddec_M_P1)
M_Ned_t = scipy_to_torch_sparse(ddec_M_Ned)

def coarse_M0(W):
    return W.T @ torch.sparse.mm(M_P1_t, W)        # (N0, N0)

def coarse_M1(W, lo_unique, hi_unique, pairs_i, pairs_j):
    Wlo = W[lo_unique]      # (n_fine_edges, N0)
    Whi = W[hi_unique]      # (n_fine_edges, N0)

    # C[(p,q), e] realizes (W tensor W) for coarse pair (p,q) against fine edge e
    C = (Wlo[:, pairs_i]*Whi[:, pairs_j] - Wlo[:, pairs_j]*Whi[:, pairs_i]).T   # (45, n_fine_edges)

    return C @ torch.sparse.mm(M_Ned_t, C.T)        # (45, 45)


In [419]:
M0 = coarse_M0(W)
M1 = coarse_M1(W, lo_unique, hi_unique, pairs_i, pairs_j)

print("M0 shape:", M0.shape, " symmetric:", torch.allclose(M0, M0.T, atol=1e-5))
print("M1 shape:", M1.shape, " symmetric:", torch.allclose(M1, M1.T, atol=1e-5))
print("M0 positive diagonal:", (torch.diag(M0) > 0).all().item())
print("M1 positive diagonal:", (torch.diag(M1) > 0).all().item())


M0 shape: torch.Size([10, 10])  symmetric: True
M1 shape: torch.Size([45, 45])  symmetric: True
M0 positive diagonal: True
M1 positive diagonal: True


In [420]:
N0_int, N0_bdry = 8, 2
free_idx  = torch.arange(N0_int)
fixed_idx = torch.arange(N0_int, N0)

def assemble_residual(u_free, M1, eps_raw, log_lam, f_theta,
                      oriented_areas=None, zero_ref=None):
    """Physics residual on free DOFs: F(u_hat; theta, lambda) = 0."""
    u_hat = torch.zeros(N0, 2, dtype=u_free.dtype, device=u_free.device)
    u_hat[free_idx] = u_free.reshape(N0_int, 2)

    eps = torch.nn.functional.softplus(eps_raw)
    K = eps * (delta_0.T @ M1 @ delta_0)
    r_node = K @ u_hat - f_theta

    if oriented_areas is not None:
        flux = flux_net(
            u_hat[pairs_i], u_hat[pairs_j],
            oriented_areas, log_lam,
            zero_ref=zero_ref,
        )
        r_node = r_node + delta_0.T @ M1 @ flux

    return r_node[free_idx].reshape(-1)


In [421]:
def newton_solve(M1, eps_raw, log_lam, f_theta,
                 oriented_areas=None, zero_ref=None,
                 u_free_init=None, tol=1e-6, max_iter=50, damping=0.0):
    """Newton solve for assemble_residual(u_free; ...) = 0."""
    n = N0_int * 2
    device, dtype = M1.device, M1.dtype

    if u_free_init is None:
        u_free = torch.zeros(n, dtype=dtype, device=device)
    else:
        u_free = u_free_init.detach().clone()

    def residual_fn(x):
        return assemble_residual(
            x, M1, eps_raw, log_lam, f_theta, oriented_areas, zero_ref
        )

    I = torch.eye(n, dtype=dtype, device=device)
    eta = 1.0
    old_u_free = u_free.clone()
    old_resid_norm = float('inf')
    converged = False
    resid_norm = float('inf')
    J = None

    with torch.no_grad():
        for it in range(max_iter):
            J = torch.func.jacrev(residual_fn)(u_free)
            r = residual_fn(u_free)
            resid_norm = r.norm().item()

            if resid_norm < tol:
                converged = True
                break

            if it > 0:
                if resid_norm < old_resid_norm:
                    eta = 1.0
                else:
                    eta *= 0.5
                    u_free = old_u_free.clone()
                    J = torch.func.jacrev(residual_fn)(u_free)
                    r = residual_fn(u_free)
                    resid_norm = r.norm().item()

            rhs = -r
            if damping > 0:
                delta = torch.linalg.solve(J + damping * I, rhs)
            else:
                delta = torch.linalg.solve(J, rhs)

            old_u_free = u_free.clone()
            old_resid_norm = resid_norm
            u_free = (u_free + eta * delta).detach()

    u_hat = torch.zeros(N0, 2, dtype=dtype, device=device)
    u_hat[free_idx] = u_free.reshape(N0_int, 2)
    return u_hat, resid_norm, it, converged, u_free, J


In [422]:
eps_raw = torch.tensor(0.0, requires_grad=True)

log_lam_val = torch.tensor(np.log10(1000.0), dtype=torch.float32)

f_theta_test = torch.zeros(N0, 2)
f_theta_test[2:5] = torch.tensor([0.0, -1.0])

# dummy geometry if not already defined
oriented_areas_smoke = torch.randn(n_pairs, 2)

# compute zero reference (required by new solver)
with torch.no_grad():
    u0_i = torch.zeros((n_pairs, 2))
    u0_j = torch.zeros((n_pairs, 2))

    zero_ref = flux_net._f(
        u0_i, u0_j, oriented_areas_smoke, log_lam_val
    ).detach()

u_hat, resid_norm, n_iter, converged, u_free, J = newton_solve(
    M1, eps_raw, log_lam_val, f_theta_test,
    oriented_areas=oriented_areas_smoke, zero_ref=zero_ref,
)

print(f"converged={converged}, iterations={n_iter}, final residual norm={resid_norm:.2e}")
print("u_hat:", u_hat)

converged=False, iterations=49, final residual norm=1.54e-05
u_hat: tensor([[ 0.0081, -0.0336],
        [-0.0130, -0.0645],
        [-0.0359, -0.0861],
        [-0.0555, -0.0873],
        [-0.0661, -0.0719],
        [-0.0662, -0.0473],
        [-0.0586, -0.0244],
        [-0.0461, -0.0066],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000]])


In [423]:
W_synthetic = torch.zeros(len(ddec_pts), N0)
x, y = ddec_pts[:,0], ddec_pts[:,1]
is_bdry_np = (x < 1e-12)

interior_rows = np.where(~is_bdry_np)[0]
band = np.clip((x[interior_rows] * N0_int).astype(int), 0, N0_int-1)
W_synthetic[interior_rows, band] = 1.0          # 8 vertical bands, interior

bdry_rows = np.where(is_bdry_np)[0]
bdry_band = np.clip((y[bdry_rows] * N0_bdry).astype(int), 0, N0_bdry-1)
W_synthetic[bdry_rows, N0_int + bdry_band] = 1.0   # 2 bands, boundary

print("row sums:", W_synthetic.sum(dim=1).min().item(), W_synthetic.sum(dim=1).max().item())

row sums: 1.0 1.0


In [424]:
M1_synth = coarse_M1(W_synthetic, lo_unique, hi_unique, pairs_i, pairs_j)

K_synth = torch.nn.functional.softplus(eps_raw).detach() * (delta_0.T @ M1_synth.detach() @ delta_0)
print("condition number with synthetic W:", torch.linalg.cond(K_synth[free_idx][:,free_idx]).item())

u_hat_s, resid_s, n_iter_s, converged_s, _, _ = newton_solve(M1_synth, eps_raw, torch.tensor(np.log10(1000.0), dtype=torch.float32), f_theta_test, damping=1e-3)
print(f"converged={converged_s}, iterations={n_iter_s}, residual={resid_s:.2e}")
print(u_hat_s)

condition number with synthetic W: 113.49586486816406
converged=False, iterations=49, residual=1.58e-06
tensor([[ 0.0000, -0.0341],
        [ 0.0000, -0.0682],
        [ 0.0000, -0.1022],
        [ 0.0000, -0.1250],
        [ 0.0000, -0.1363],
        [ 0.0000, -0.1363],
        [ 0.0000, -0.1363],
        [ 0.0000, -0.1363],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000]])


In [425]:
K_coarse = torch.nn.functional.softplus(eps_raw).detach() * (delta_0.T @ M1_synth.detach() @ delta_0)   # (10,10)
K_free = K_coarse[free_idx][:, free_idx]                                                          # (8,8)

print("condition number:", torch.linalg.cond(K_free).item())
print("eigenvalues:", torch.linalg.eigvalsh(K_free))

u_free_linear = torch.linalg.solve(K_free, f_theta_test[free_idx])
print("linear-only solution (no Newton, no nonlinear flux):")
print(u_free_linear)


condition number: 113.49586486816406
eigenvalues: tensor([  2.9977,  26.3707,  69.9599, 127.8784, 192.3041, 254.5358, 306.1689,
        340.2299])
linear-only solution (no Newton, no nonlinear flux):
tensor([[ 0.0000, -0.0341],
        [ 0.0000, -0.0682],
        [ 0.0000, -0.1022],
        [ 0.0000, -0.1250],
        [ 0.0000, -0.1363],
        [ 0.0000, -0.1363],
        [ 0.0000, -0.1363],
        [ 0.0000, -0.1363]])


In [426]:
def get_u_target(solve):
    U = solve['U']
    Ux, Uy = U[0::2][:len(ddec_pts)], U[1::2][:len(ddec_pts)]   # corner dofs only
    return torch.tensor(np.stack([Ux, Uy], axis=1), dtype=torch.float32)   # (n_fine, 2)

In [427]:
M_P1_rowsum = torch.tensor(np.asarray(ddec_M_P1.sum(axis=1)).ravel(), dtype=torch.float32)   # (n_fine,)

def get_f_theta(solve, W):
    fx0, fy0 = solve['load']
    f_fine = M_P1_rowsum[:,None] * torch.tensor([fx0, fy0], dtype=torch.float32)   # (n_fine, 2)
    return W.T @ f_fine                                                            # (N0, 2)


In [428]:
solve0 = ddec_solves[0]
log_lam = torch.tensor(np.log10(ddec_lam), dtype=torch.float32)

W_real = pou_net(xy, log_lam, is_bdry)
M1_real = coarse_M1(W_real, lo_unique, hi_unique, pairs_i, pairs_j)
f_theta_real = get_f_theta(solve0, W_real)
u_target = get_u_target(solve0)

u_hat_real, resid_real, n_iter_real, converged_real,_,_ = newton_solve(M1_real, eps_raw, log_lam, f_theta_real)
print(f"converged={converged_real}, residual={resid_real:.2e}")

u_recon = W_real @ u_hat_real          # lift coarse solution back to fine resolution
print("reconstruction error vs target:", (u_recon - u_target).norm().item(), " | target norm:", u_target.norm().item())


converged=True, residual=8.53e-07
reconstruction error vs target: 68.41627502441406  | target norm: 73.20780181884766


In [429]:
def adjoint_solve(u_free_converged, J_free, W, u_target):
    """
    Adjoint / KKT multiplier solve:
        J^T lam_dual = -dL/du_free
    with L_data = 1/2 ||W @ u_hat(u_free) - u_target||^2
    """
    def data_loss_fn(u_free):
        u_hat = torch.zeros(N0, 2, dtype=u_free.dtype, device=u_free.device)
        u_hat[free_idx] = u_free.reshape(N0_int, 2)
        recon = W @ u_hat - u_target
        return 0.5 * recon.pow(2).sum()
    dL_du_free = torch.func.grad(data_loss_fn)(u_free_converged)
    lam_dual = torch.linalg.solve(J_free.T, -dL_du_free)
    return lam_dual


In [430]:
def parameter_gradient(u_free_converged, lam_dual, M1, eps_raw, log_lam, f_theta, params,oriented_areas=None, zero_ref=None):
    r = assemble_residual(u_free_converged.detach(), M1, eps_raw, log_lam, f_theta,oriented_areas, zero_ref)
    pseudo_loss = (lam_dual.detach() * r).sum()
    grads = torch.autograd.grad(pseudo_loss, params, allow_unused=True)
    return [g if g is not None else torch.zeros_like(p) for g, p in zip(grads, params)]


In [431]:
solve_example = ddec_solves[1]
print("load case:", solve_example['load'])

log_lam = torch.tensor(np.log10(ddec_lam), dtype=torch.float32)
W = pou_net(xy, log_lam, is_bdry)
M1 = coarse_M1(W, lo_unique, hi_unique, pairs_i, pairs_j)

f_theta = get_f_theta(solve_example, W)
u_target = get_u_target(solve_example)

u_hat, resid_norm, n_iter, converged, u_free, J = newton_solve(
    M1, eps_raw, log_lam, f_theta,
    oriented_areas=oriented_areas, zero_ref=zero_ref,
)
print(f"forward solve: converged={converged}, residual={resid_norm:.2e}, iterations={n_iter}")

lam_dual = adjoint_solve(u_free, J, W, u_target)
print("adjoint multiplier norm:", lam_dual.norm().item())

params = list(pou_net.parameters()) + list(flux_net.parameters()) + [eps_raw]
grads = parameter_gradient(
    u_free, lam_dual, M1, eps_raw, log_lam, f_theta, params,
    oriented_areas=oriented_areas, zero_ref=zero_ref,
)
print("any NaNs in gradients?:", any(torch.isnan(g).any().item() for g in grads))
print("parameter gradient norms (first 5):", [g.norm().item() for g in grads[:5]])

load case: (np.float64(0.0), np.float64(-2.0))
forward solve: converged=False, residual=3.74e-05, iterations=49
adjoint multiplier norm: 851.7932739257812
any NaNs in gradients?: False
parameter gradient norms (first 5): [0.0, 0.0, 0.0, 0.0, 943.4841918945312]


In [432]:
# TRAIN_V5 — reload after any edit to ddec_train.py
import importlib
import ddec_train
importlib.reload(ddec_train)
from ddec_train import training_step, train
print("loaded", ddec_train.__file__)


loaded /Users/tsnevan/Desktop/pimi/ddec_train.py


In [433]:
# recon_error = L2 difference between the coarse model's prediction and the high-fidelity FEM displacement field
# resid = norm of the leftover PDE residual r = F(û;θ,λ) − f_θ at wherever Newton's iteration stopped
# grad_norm = the pre-clip norm of the raw parameter gradient

In [435]:
import copy
import importlib
import ddec_train
importlib.reload(ddec_train)
from ddec_train import train

pou_net = POUGenerator(n_interior=N0_int, n_boundary=N0_bdry)
flux_net = FluxCorrectionTransformer(num_fields=2, problem_dim=2, num_input_params=1,
                                    latent_dim=16, num_heads=4, num_blocks=2)
eps_raw = torch.tensor(0.0, requires_grad=True)

params = list(pou_net.parameters()) + list(flux_net.parameters()) + [eps_raw]
optimizer = torch.optim.Adam(params, lr=1e-4)
log_lam = torch.tensor(np.log10(ddec_lam), dtype=torch.float32)

log_lam_val = torch.tensor(np.log10(1000.0), dtype=torch.float32)
W = pou_net(xy, log_lam_val, is_bdry)
M1 = coarse_M1(W, lo_unique, hi_unique, pairs_i, pairs_j)
f_theta_test = torch.zeros(N0, 2)
f_theta_test[2:5] = torch.tensor([0.0, -1.0])

u_hat, resid, n_iter, converged, u_free, J = newton_solve(
    M1, eps_raw, log_lam_val, f_theta_test,
    oriented_areas=None, zero_ref=None,
    tol=2e-4, max_iter=80,
)
print(f"smoke test: converged={converged}, iters={n_iter}, resid={resid:.2e}")

n_epochs = 200
history, best_state, best_recon = train(
    n_epochs, optimizer, params, eps_raw, log_lam, ddec_solves,
    flux_net=flux_net,
    pou_net=pou_net,
    xy=xy,
    is_bdry=is_bdry,
    coarse_M1=coarse_M1,
    compute_oriented_areas=compute_oriented_areas,
    get_f_theta=get_f_theta,
    get_u_target=get_u_target,
    assemble_residual=assemble_residual,
    newton_solve=newton_solve,
    adjoint_solve=adjoint_solve,
    lo_unique=lo_unique,
    hi_unique=hi_unique,
    pairs_i=pairs_i,
    pairs_j=pairs_j,
    free_idx=free_idx,
    delta_0=delta_0,
    log_every=10,
)


smoke test: converged=True, iters=1, resid=3.80e-06
epoch    0  [linear-Newton pw=3.33e-04]  recon_err=85.531  resid=1.11e-06  grad_norm=0.00  iters=1.0  skipped=0/4  stepped=True  eps=0.6931  cond=1.04e+02
epoch   10  [nonlinear pw=3.67e-03]  recon_err=85.139  resid=8.50e-05  grad_norm=4.57  iters=1.8  skipped=0/4  stepped=True  eps=0.6926  cond=1.04e+02
epoch   20  [nonlinear pw=7.00e-03]  recon_err=85.158  resid=9.08e-05  grad_norm=8.79  iters=1.0  skipped=0/4  stepped=True  eps=0.6923  cond=1.04e+02
epoch   30  [nonlinear pw=1.00e-02]  recon_err=85.173  resid=9.09e-05  grad_norm=12.68  iters=1.0  skipped=0/4  stepped=True  eps=0.6920  cond=1.04e+02
epoch   40  [nonlinear pw=1.00e-02]  recon_err=85.194  resid=9.43e-05  grad_norm=12.83  iters=1.0  skipped=0/4  stepped=True  eps=0.6918  cond=1.04e+02
epoch   50  [nonlinear pw=1.00e-02]  recon_err=85.220  resid=9.87e-05  grad_norm=13.03  iters=1.0  skipped=0/4  stepped=True  eps=0.6916  cond=1.04e+02
epoch   60  [nonlinear pw=1.00e-02]

KeyboardInterrupt: 